# Mission 2 — Final Full-Training Refit + Official Validation

고정된 Stage 14 계약을 전체 organizer Training에 refit하고 Official Validation에서 한 번 평가합니다. 모델은 TabM, Tabular ResNet, GANDALF이며 149D `Clean59 + F01 + F02`, seed 42, E_TRG 1:1:1, threshold 0.5를 유지합니다. Hyperparameter/weight/threshold 탐색과 calibration은 없습니다.

**순서:** code bundle SHA 확인 → Stage 14/Training preflight → Official Validation feature alignment → Training-only scaler 및 고정 epoch refit → checkpoint reload → Official 평가 → 저장 prediction 재계산/audit.

실행 코드는 로컬 bundle 업로드로 고정합니다. `data/raw_original/Validation`의 WAV/JSON과 `data/raw_original/대학부 데이터` Training archive만 읽고, 원본 데이터는 bundle에 포함하지 않습니다. 실패한 고유 run은 재사용하거나 덮지 않습니다.


In [ ]:
from pathlib import Path
import hashlib
import json
import shutil
import sys
import zipfile
from google.colab import drive, files

drive.mount('/content/drive', force_remount=False)
MYDRIVE_ROOT = Path('/content/drive/MyDrive').resolve(strict=True)
PROJECT_SHORTCUT = MYDRIVE_ROOT / 'prelim'
if not PROJECT_SHORTCUT.is_dir():
    raise FileNotFoundError(f'MyDrive/prelim 프로젝트 바로가기를 찾지 못했습니다: {PROJECT_SHORTCUT}')
REPO_ROOT = PROJECT_SHORTCUT.resolve(strict=True)

CODE_BUNDLE_NAME = 'mission2_final_refit_code_bundle_unicodefix.zip'
EXPECTED_BUNDLE_SHA256 = '02dc4983998ea91f46395874f68d0093feb8caa74652a623e569160b02d3b61e'
BUNDLE_PATH = Path('/content') / CODE_BUNDLE_NAME
if not BUNDLE_PATH.is_file():
    drive_bundle = MYDRIVE_ROOT / 'Colab Notebooks' / CODE_BUNDLE_NAME
    if drive_bundle.is_file():
        shutil.copy2(drive_bundle, BUNDLE_PATH)
    else:
        uploaded = files.upload()
        if CODE_BUNDLE_NAME not in uploaded:
            raise FileNotFoundError(f'업로드 파일 이름이 {CODE_BUNDLE_NAME!r}인지 확인하세요.')
        BUNDLE_PATH.write_bytes(uploaded[CODE_BUNDLE_NAME])
observed_bundle_sha256 = hashlib.sha256(BUNDLE_PATH.read_bytes()).hexdigest()
if observed_bundle_sha256 != EXPECTED_BUNDLE_SHA256:
    raise ValueError(f'Code bundle SHA-256 불일치: expected={EXPECTED_BUNDLE_SHA256}, observed={observed_bundle_sha256}')

with zipfile.ZipFile(BUNDLE_PATH) as archive:
    bundle_manifest = json.loads(archive.read('bundle_manifest.json').decode('utf-8'))
    if bundle_manifest.get('schema_version') != 'mission2_final_refit_code_bundle_v1' or bundle_manifest.get('contains_raw_data') is not False:
        raise ValueError('Code bundle manifest 계약이 다릅니다.')
    declared = bundle_manifest.get('files', {})
    actual = set(archive.namelist()) - {'bundle_manifest.json'}
    if actual != set(declared):
        raise ValueError('Code bundle file list와 내부 SHA 목록이 다릅니다.')
    for relative, expected_sha in declared.items():
        content = archive.read(relative)
        if hashlib.sha256(content).hexdigest() != expected_sha:
            raise ValueError(f'Code bundle member SHA 불일치: {relative}')

BUNDLE_ROOT = Path('/content') / f'mission2_final_refit_bundle_{EXPECTED_BUNDLE_SHA256[:12]}'
if not BUNDLE_ROOT.exists():
    BUNDLE_ROOT.mkdir()
    with zipfile.ZipFile(BUNDLE_PATH) as archive:
        for relative in sorted(declared):
            target = (BUNDLE_ROOT / relative).resolve()
            if not target.is_relative_to(BUNDLE_ROOT.resolve()):
                raise ValueError(f'안전하지 않은 bundle 경로: {relative}')
            target.parent.mkdir(parents=True, exist_ok=True)
            target.write_bytes(archive.read(relative))
for relative, expected_sha in declared.items():
    extracted = BUNDLE_ROOT / relative
    if not extracted.is_file() or hashlib.sha256(extracted.read_bytes()).hexdigest() != expected_sha:
        raise ValueError(f'Extracted code bundle member changed: {relative}')
print('Verified code bundle:', BUNDLE_PATH)
print('SHA-256:', observed_bundle_sha256)
print('Source files:', len(declared), '| raw data included: no')


## Colab 패키지

Stage 14에서 사용한 모델/전처리 라이브러리 버전을 고정합니다. 설치나 import 오류가 있으면 학습을 시작하지 마세요.

처음 패키지 설치로 NumPy가 바뀐 뒤 import 오류가 나면 런타임 메뉴에서 **세션 다시 시작 및 모두 실행**을 선택하세요. 새 커널이 설치된 패키지를 다시 읽은 뒤 계약 테스트부터 진행합니다.


In [ ]:
%pip -q install numpy==2.5.2 librosa==1.0.0 opensmile==2.6.0 tabm==0.0.3 rtdl-revisiting-models==0.0.2 rtdl-num-embeddings==0.0.12 pytorch-tabular==1.2.0 pyarrow pytest


## A. Bundle 계약 테스트 및 GPU 확인

테스트는 synthetic/unit contract만 검사하며 실제 데이터 학습을 시작하지 않습니다. 최종 refit은 CUDA가 있을 때만 시작합니다.


In [ ]:
import os
import subprocess
import numpy as np
import pandas as pd
import torch

sys.path.insert(0, str(BUNDLE_ROOT / 'src'))
from dcc_data import mission2_acoustic_feature_grid as grid
from dcc_data import mission2_stage14_ensemble as stage14
from dcc_data import mission2_final_fulltrain_refit as final
from dcc_data import mission2_model_family_screen as screen
from dcc_data import mission2_model_experiments as models
from dcc_data.mission2_egemaps_colab import prepare_shared_training_raw

TEST_FILE = BUNDLE_ROOT / 'tests' / 'test_mission2_final_fulltrain_refit.py'
test_env = os.environ.copy()
bundle_source_path = str(BUNDLE_ROOT / 'src')
test_env['PYTHONPATH'] = os.pathsep.join(
    value for value in (bundle_source_path, test_env.get('PYTHONPATH')) if value
)
test_run = subprocess.run(
    [sys.executable, '-m', 'pytest', '-q', '--basetemp', '/content/mission2_final_refit_pytest', str(TEST_FILE)],
    text=True, capture_output=True, check=False, env=test_env,
)
print(test_run.stdout)
if test_run.stderr:
    print(test_run.stderr)
if test_run.returncode != 0:
    raise RuntimeError('Final-refit contract tests failed; no model fit was started.')
if not torch.cuda.is_available():
    raise RuntimeError('CUDA GPU가 없습니다. CPU로 바꾸지 않고 여기서 중단합니다.')
DEVICE = 'cuda'
print('GPU:', torch.cuda.get_device_name())
print('CUDA build:', torch.version.cuda)

with zipfile.ZipFile(BUNDLE_PATH) as bundle_archive:
    bundle_manifest_sha256 = hashlib.sha256(bundle_archive.read('bundle_manifest.json')).hexdigest()

code_bundle_provenance = {
    'bundle_filename': CODE_BUNDLE_NAME,
    'bundle_sha256': observed_bundle_sha256,
    'bundle_manifest_sha256': bundle_manifest_sha256,
    'source_file_hashes': declared,
    'executed_final_module': str(Path(final.__file__).resolve()),
    'executed_final_module_sha256': hashlib.sha256(Path(final.__file__).read_bytes()).hexdigest(),
}


## A. Stage 14·Training 계약 preflight

Training은 기존 고정 internal split이 아니라 organizer 전체 Training 889,358행을 사용합니다. Stage 14의 저장 config, checkpoint, history와 149D cache alignment를 대조합니다. Official Validation은 이 단계에서 feature/label manifest 정합성 검증 외 모델 선택에 사용하지 않습니다.


In [ ]:
LAYOUT = grid.resolve_colab_drive_layout(MYDRIVE_ROOT)
SOURCE_FEATURE_RUN_ROOT = LAYOUT['source_run_root']
STAGE6_ROOT = REPO_ROOT / grid.STAGE6_REFERENCE_RELATIVE
EGEMAPS_88D = REPO_ROOT / grid.EGEMAPS_88D_RELATIVE
TRANSFER_IDENTITY = REPO_ROOT / 'tmp/mission2_egemaps_transfer_identity.json'
for required in (SOURCE_FEATURE_RUN_ROOT / 'feature_cache' / 'manifest.json',
                 STAGE6_ROOT / 'contract.json', EGEMAPS_88D, TRANSFER_IDENTITY):
    if not required.is_file():
        raise FileNotFoundError(f'필수 Training-only artifact가 없습니다: {required}')

cache_paths = models.standalone_training_cache_paths(REPO_ROOT)
for required in (cache_paths.manifest_parquet, cache_paths.manifest_metadata):
    if not required.is_file():
        raise FileNotFoundError(f'Training source manifest가 없습니다: {required}')

archive_candidates = [
    REPO_ROOT / 'data' / 'raw_original' / '대학부 데이터',
    REPO_ROOT / 'data' / 'raw_original' / '대학부데이터',
]
archive_root = None
for candidate_root in archive_candidates:
    archive_paths = screen.shared_training_archive_paths(candidate_root)
    if candidate_root.is_dir() and all(path.is_file() for path in archive_paths):
        archive_root = candidate_root
        break
if archive_root is None:
    raise FileNotFoundError('data/raw_original/대학부 데이터(또는 대학부데이터)에서 12개 Training ZIP을 찾지 못했습니다.')

training_raw_candidates = [Path('/content/mission2_stage13_training_raw'), Path('/content/mission2_grid_training_raw')]
TRAINING_RAW_ROOT = next((path for path in training_raw_candidates if path.is_dir()), None)
if TRAINING_RAW_ROOT is None:
    TRAINING_RAW_ROOT = Path('/content/mission2_final_training_raw')
    training_manifest = pd.read_parquet(
        cache_paths.manifest_parquet,
        columns=['split', 'call_id', 'audio_path', 'json_path'],
        filters=[('split', '==', 'Training')],
    )
    staging_audit = prepare_shared_training_raw(
        training_manifest, screen.shared_training_archive_paths(archive_root), TRAINING_RAW_ROOT,
    )
    print('Training staging:', json.dumps(staging_audit, ensure_ascii=False))
    del training_manifest

validation_root_candidates = [REPO_ROOT / 'data' / 'raw_original', archive_root]
OFFICIAL_RAW_ROOT = next((root for root in validation_root_candidates if (root / 'Validation').is_dir()), None)
if OFFICIAL_RAW_ROOT is None:
    raise FileNotFoundError('data/raw_original 또는 대학부 데이터 archive에서 Validation 폴더를 찾지 못했습니다.')
PRIOR_RUN_ROOTS = grid.stage10_to13_prior_run_roots(REPO_ROOT)
context = grid.load_training_context(
    REPO_ROOT, TRAINING_RAW_ROOT, prior_run_roots=PRIOR_RUN_ROOTS,
    stage6_root=STAGE6_ROOT, egemaps_cache_path=EGEMAPS_88D,
    transfer_identity_path=TRANSFER_IDENTITY,
)
training_matrix, feature_names, training_audit = final.load_full_training_matrix(
    context, SOURCE_FEATURE_RUN_ROOT,
)
if training_matrix.shape != (889_358, 149) or len(np.unique(context.call_ids)) != 29_200:
    raise ValueError('전체 organizer Training 889,358행/29,200통화가 아닙니다.')

stage14_candidates = sorted(
    path for path in LAYOUT['output_root'].glob(f'{SOURCE_FEATURE_RUN_ROOT.name}{stage14.RUN_SUFFIX}*')
    if path.is_dir() and (path / 'contract.json').is_file()
)
complete_stage14 = []
for candidate_root in stage14_candidates:
    ledger = json.loads((candidate_root / 'run_ledger.json').read_text(encoding='utf-8'))
    if ledger.get('status') != 'ENSEMBLES_COMPLETE' or ledger.get('ensemble_candidates', {}).get('E_TRG') != 'COMPLETE':
        continue
    evidence = final.load_stage14_refit_evidence(
        context, candidate_root, SOURCE_FEATURE_RUN_ROOT,
    )
    complete_stage14.append((candidate_root, evidence))
if len(complete_stage14) != 1:
    raise RuntimeError(f'완전 검증된 E_TRG Stage 14 run이 정확히 하나여야 합니다. 후보: {[str(x[0]) for x in complete_stage14]}')
STAGE14_RUN_ROOT, stage14_evidence = complete_stage14[0]
print('Training raw:', TRAINING_RAW_ROOT)
print('Official raw:', OFFICIAL_RAW_ROOT)
print('Stage 14 selected by completed E_TRG ledger:', STAGE14_RUN_ROOT)
print('149D schema:', training_audit['feature_schema_fingerprint'])
print('Fixed epochs:', stage14_evidence['fixed_refit_epochs'])
print('Training/cache preflight: PASS')


## A·D. Official Validation manifest 및 frozen 149D feature alignment

이 셀은 Official Validation 3,640개 JSON에서 JSON 배열 위치와 `startAt`, `endAt`, `speaker`만 추적용으로 보존합니다. transcript는 모델 입력에 사용하거나 별도 저장하지 않습니다. 기존 MFCC40은 exact identity가 맞을 때 재사용하고, 누락된 Clean59/F01/F02 블록만 동일 코드로 추출합니다.


In [ ]:
OUTPUT_PARENT = MYDRIVE_ROOT / 'mission2_final_fulltrain_runs'
RUN_ROOT = final.create_run_root(OUTPUT_PARENT)
validation_source = final.build_official_validation_manifest(OFFICIAL_RAW_ROOT, RUN_ROOT)
MFCC_CACHE = REPO_ROOT / 'data/processed/mission2/validation_mfcc20_nfft512_hop160_v1.npz'
MFCC_METADATA = REPO_ROOT / 'data/processed/mission2/validation_mfcc20_nfft512_hop160_v1.json'

def extraction_progress(block, completed, total):
    if completed % 250 == 0 or completed == total:
        print(f'{block}: {completed:,}/{total:,}', flush=True)

validation_data = final.load_or_extract_official_validation_features(
    validation_source['manifest'], OFFICIAL_RAW_ROOT,
    source_manifest_path=validation_source['path'],
    source_manifest_metadata_path=validation_source['metadata_path'],
    mfcc_cache_path=MFCC_CACHE, mfcc_metadata_path=MFCC_METADATA,
    run_root=RUN_ROOT, worker_count=4, on_call_done=extraction_progress,
)
if (validation_data['X'].shape != (111_947, 149)
        or validation_data['manifest'].call_id.nunique() != 3_640
        or tuple(validation_data['feature_names']) != tuple(feature_names)):
    raise ValueError('Official Validation 111,947행의 149D feature order/alignment가 다릅니다.')

contract = final.initialize_final_run(
    RUN_ROOT, context, training_matrix, feature_names, training_audit,
    validation_data, stage14_evidence,
    code_bundle_provenance=code_bundle_provenance,
)
environment = final.record_environment(RUN_ROOT, device=DEVICE)
scaler, X_train_scaled, X_validation_scaled = final.fit_full_training_scaler(
    training_matrix, validation_data['X'],
)
final._save_full_training_scaler(scaler, RUN_ROOT, feature_names)
print('Final run:', RUN_ROOT)
print('Training:', training_matrix.shape, '| Official Validation:', validation_data['X'].shape)
print('Scaler fit rows:', scaler.n_samples_seen_)
print('Fixed epochs:', contract['fixed_refit_epochs'])
print('Code bundle:', code_bundle_provenance['bundle_sha256'])
print('Refit preflight: PASS; Official Validation was not used for fit/selection.')


## B·C. 전체 Training refit 및 checkpoint reload

모델별 저장 best Internal epoch만 학습 길이로 사용합니다. Official Validation은 forward prediction 전까지 scaler, gradient, loss, early stopping, checkpoint selection에 전달하지 않습니다. 순서는 TabM → Tabular ResNet → GANDALF이며 첫 실패에서 중단됩니다.


In [ ]:
def record_epoch(model_id, epoch, total_epochs, train_loss):
    member_id = f'{model_id}_F01_F02'
    progress_path = RUN_ROOT / 'members' / member_id / 'epoch_progress.jsonl'
    row = {'model_id': model_id, 'epoch': int(epoch), 'total_epochs': int(total_epochs),
           'train_loss': float(train_loss), 'official_validation_used': False}
    with progress_path.open('a', encoding='utf-8') as stream:
        stream.write(json.dumps(row, allow_nan=False) + '\n')
        stream.flush()
    print(f'{model_id}: epoch {epoch}/{total_epochs}, train_loss={train_loss:.6f}', flush=True)

fit_result = final.run_remaining_final_members(
    RUN_ROOT, X_train_scaled, context.y, X_validation_scaled,
    validation_data, feature_names, device=DEVICE, on_epoch_end=record_epoch,
)
if fit_result['status'] != 'ALL_MEMBERS_COMPLETE':
    raise RuntimeError(f'모든 모델이 완료되지 않았습니다: {fit_result}')
print(json.dumps(fit_result, ensure_ascii=False, indent=2))


## D·E·F. Official Validation 평가, 저장 결과 재계산 및 artifact audit

세 단독 모델과 E_TRG를 같은 111,947행에서 평가합니다. threshold는 0.5로 고정합니다. 최종 JSON/Parquet/CSV와 체크포인트 파일이 실제로 존재하고 크기가 0보다 큰지 확인한 뒤 SHA 목록을 대조합니다.


In [ ]:
result = final.evaluate_final_run(RUN_ROOT, validation_data)
if result['status'] != 'FINAL_VERIFIED':
    raise RuntimeError(f'Final audit did not pass: {result["status"]}')

required_artifacts = [
    RUN_ROOT / 'contract.json', RUN_ROOT / 'environment.json',
    RUN_ROOT / 'feature_schema.json', RUN_ROOT / 'training_config.json',
    RUN_ROOT / 'scaler.json', RUN_ROOT / 'ensemble_predictions.parquet',
    RUN_ROOT / 'official_validation_comparison.csv',
    RUN_ROOT / 'official_validation_metrics.json', RUN_ROOT / 'final_audit.json',
    RUN_ROOT / 'artifact_manifest.json',
]
for member_id in final.MEMBER_IDS:
    folder = RUN_ROOT / 'members' / member_id
    required_artifacts.extend([
        folder / 'config.json', folder / 'scaler.json', folder / 'training_history.csv',
        folder / 'official_validation_predictions.parquet', folder / 'metrics.json',
        folder / 'artifact_manifest.json',
    ])
    if member_id == 'gandalf_F01_F02':
        required_artifacts.append(folder / 'checkpoint')
    else:
        required_artifacts.append(folder / 'checkpoint.pt')
for path in required_artifacts:
    if not path.exists() or (path.is_file() and path.stat().st_size == 0):
        raise FileNotFoundError(f'저장 artifact 검증 실패: {path}')
root_artifact_manifest = json.loads((RUN_ROOT / 'artifact_manifest.json').read_text(encoding='utf-8'))
for relative, expected_sha in root_artifact_manifest['files'].items():
    observed_sha = hashlib.sha256((RUN_ROOT / relative).read_bytes()).hexdigest()
    if observed_sha != expected_sha:
        raise ValueError(f'Drive 저장 artifact SHA 불일치: {relative}')

print('Run:', RUN_ROOT)
print('Status:', result['status'])
display(result['comparison'][['model_id','validation_accuracy','recall_0','recall_1','hce','fp','fn','official_validation_inference_seconds']])
display(result['paired_errors'][['model_id','ensemble_only_correct','member_only_correct','common_error','net_correction','changed_rows']])
display(result['slices'].loc[result['slices']['slice'].isin(['all','overlap','no_overlap','duration_0.5-1s'])])
print(json.dumps(result['audit'], ensure_ascii=False, indent=2))
print('FINAL_RUN_ARTIFACTS_AND_SHA256: PASS')


## G. E_TRG Official Validation 오분류 분석

저장된 FINAL_VERIFIED 예측과 text-free manifest만 읽어 FP/FN, confidence, duration/overlap slice, 멤버 투표를 집계합니다. threshold는 0.5로 고정합니다. 재학습·재추출·음성 청취·transcript 접근은 하지 않습니다.

개별 오류행은 run의 Drive 분석 폴더에 저장하고 이 Notebook에는 출력으로 포함하지 않습니다. 집계 요약은 docs/mission2_experiment_log.md에 기록합니다.

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=False)

# Official Validation 오분류 분석 (기존 저장 예측만 읽음)
# 범위: E_TRG FP/FN, confidence, duration/overlap, 세 멤버 투표 불일치.
# 재학습·재추출·threshold 조정·음성 청취·transcript 접근은 하지 않습니다.
from pathlib import Path
import json
import sys
import numpy as np
import pandas as pd

ANALYSIS_RUN_ID = "mission2_final_fulltrain_official_validation_20261012T151108Z_a1606771"
ANALYSIS_RUN_ROOT = Path("/content/drive/MyDrive/mission2_final_fulltrain_runs") / ANALYSIS_RUN_ID
if "RUN_ROOT" in globals() and Path(RUN_ROOT).name == ANALYSIS_RUN_ID:
    ANALYSIS_RUN_ROOT = Path(RUN_ROOT)
if not ANALYSIS_RUN_ROOT.is_dir():
    raise FileNotFoundError(f"저장된 Official Validation run을 찾지 못했습니다: {ANALYSIS_RUN_ROOT}. Drive가 연결됐는지 확인하세요.")

# FINAL_VERIFIED audit와 source manifest provenance를 먼저 확인합니다.
analysis_audit = json.loads((ANALYSIS_RUN_ROOT / "final_audit.json").read_text(encoding="utf-8"))
source_meta = json.loads((ANALYSIS_RUN_ROOT / "features" / "official_validation_manifest.json").read_text(encoding="utf-8"))
if analysis_audit.get("status") != "FINAL_VERIFIED" or source_meta.get("source_split") != "Validation":
    raise RuntimeError("FINAL_VERIFIED Official Validation provenance가 확인되지 않습니다.")
if analysis_audit.get("fixed_threshold") != 0.5 or analysis_audit.get("official_validation_used_for_scaler_gradient_early_stopping_or_selection") is not False:
    raise RuntimeError("고정 threshold 또는 Validation 미사용 학습 계약이 예상과 다릅니다.")

analysis_manifest = pd.read_parquet(ANALYSIS_RUN_ROOT / "features" / "official_validation_manifest.parquet")
analysis_ensemble = pd.read_parquet(ANALYSIS_RUN_ROOT / "ensemble_predictions.parquet")
analysis_members = {
    "tabm": "tabm_F01_F02",
    "resnet": "tabular_resnet_F01_F02",
    "gandalf": "gandalf_F01_F02",
}
expected_keys = analysis_manifest["record_key"].astype(str).to_numpy()
expected_calls = analysis_manifest["call_id"].astype(str).to_numpy()
expected_y = analysis_manifest["speaker"].to_numpy(dtype=np.int64)

def check_prediction_alignment(frame, name):
    if len(frame) != len(analysis_manifest):
        raise ValueError(f"{name}: Validation row count 불일치")
    if (not np.array_equal(frame["record_key"].astype(str).to_numpy(), expected_keys)
            or not np.array_equal(frame["call_id"].astype(str).to_numpy(), expected_calls)
            or not np.array_equal(frame["y_true"].to_numpy(dtype=np.int64), expected_y)):
        raise ValueError(f"{name}: record/call/label 정렬 불일치")
    probability = frame["probability_speaker1"].to_numpy(dtype=np.float64)
    if not np.isfinite(probability).all() or np.any((probability < 0) | (probability > 1)):
        raise ValueError(f"{name}: 잘못된 확률값")
    if not np.array_equal(frame["prediction"].to_numpy(dtype=np.int64), (probability >= 0.5).astype(np.int64)):
        raise ValueError(f"{name}: 고정 0.5 threshold 재현 실패")

check_prediction_alignment(analysis_ensemble, "E_TRG")
analysis = analysis_ensemble.copy().reset_index(drop=True)
for short_name, member_id in analysis_members.items():
    member_path = ANALYSIS_RUN_ROOT / "members" / member_id / "official_validation_predictions.parquet"
    member_frame = pd.read_parquet(member_path)
    check_prediction_alignment(member_frame, member_id)
    analysis[f"p_{short_name}"] = member_frame["probability_speaker1"].to_numpy(dtype=np.float64)
    analysis[f"pred_{short_name}"] = member_frame["prediction"].to_numpy(dtype=np.int64)

analysis["duration_sec"] = (analysis_manifest["endAt_ms"].to_numpy(dtype=np.float64)
                             - analysis_manifest["startAt_ms"].to_numpy(dtype=np.float64)) / 1000.0
if not np.isfinite(analysis["duration_sec"]).all() or (analysis["duration_sec"] <= 0).any():
    raise ValueError("Official Validation duration metadata가 잘못됐습니다.")

# 기존 프로젝트의 overlap 정의와 duration 구간을 그대로 사용합니다.
if "BUNDLE_ROOT" in globals():
    sys.path.insert(0, str(Path(BUNDLE_ROOT) / "src"))
from dcc_data import mission2_model_experiments as analysis_models
from dcc_data import mission2_standard_representation_comparison as analysis_diagnostics
analysis["overlap"] = analysis_models._overlap_mask(analysis_manifest)
analysis["margin_from_0_5"] = np.abs(analysis["probability_speaker1"] - 0.5)
analysis["member_positive_votes"] = analysis[[f"pred_{name}" for name in analysis_members]].sum(axis=1).astype(int)
analysis["member_disagreement"] = analysis[[f"pred_{name}" for name in analysis_members]].nunique(axis=1) > 1
member_probability_columns = [f"p_{name}" for name in analysis_members]
analysis["member_probability_min"] = analysis[member_probability_columns].min(axis=1)
analysis["member_probability_max"] = analysis[member_probability_columns].max(axis=1)
analysis["member_probability_range"] = analysis["member_probability_max"] - analysis["member_probability_min"]

# 집계가 저장된 Official Validation metric과 맞는지 확인합니다.
confusion = pd.crosstab(analysis["y_true"], analysis["prediction"]).reindex(index=[0, 1], columns=[0, 1], fill_value=0)
fp = int(((analysis["y_true"] == 0) & (analysis["prediction"] == 1)).sum())
fn = int(((analysis["y_true"] == 1) & (analysis["prediction"] == 0)).sum())
wrong = int((~analysis["correct"].astype(bool)).sum())
if (len(analysis) != analysis_audit["official_validation_rows"]
        or fp != 8519 or fn != 4364 or wrong != fp + fn
        or int(confusion.to_numpy().sum()) != len(analysis)):
    raise ValueError("저장된 audit와 오분류 행 집계가 일치하지 않습니다.")

error_summary = pd.DataFrame([
    {"split": "Official Validation", "model": "E_TRG", "rows": len(analysis),
     "accuracy": float(analysis["correct"].mean()), "errors": wrong,
     "error_rate": wrong / len(analysis), "FP_true0_pred1": fp, "FN_true1_pred0": fn,
     "HCE_confidence_ge_0.90": int(((~analysis["correct"].astype(bool)) & (analysis["confidence"] >= 0.90)).sum()),
     "threshold": 0.5}
])

slice_frame = pd.DataFrame({
    "duration_sec": analysis["duration_sec"],
    "overlap": analysis["overlap"],
    "y_true": analysis["y_true"],
})
slice_rows = []
for slice_name, mask in analysis_diagnostics.slice_masks(slice_frame).items():
    mask = np.asarray(mask, dtype=bool)
    selected = analysis.loc[mask]
    n = len(selected)
    slice_rows.append({
        "slice": slice_name, "rows": n,
        "errors": int((~selected["correct"].astype(bool)).sum()),
        "error_rate": float((~selected["correct"].astype(bool)).mean()) if n else np.nan,
        "FP": int(((selected["y_true"] == 0) & (selected["prediction"] == 1)).sum()),
        "FN": int(((selected["y_true"] == 1) & (selected["prediction"] == 0)).sum()),
    })
error_slices = pd.DataFrame(slice_rows)

confidence_edges = [0.5, 0.6, 0.7, 0.8, 0.9, 1.000001]
confidence_labels = ["[0.50,0.60)", "[0.60,0.70)", "[0.70,0.80)", "[0.80,0.90)", "[0.90,1.00]"]
analysis["confidence_band"] = pd.cut(analysis["confidence"], bins=confidence_edges,
                                     labels=confidence_labels, right=False, include_lowest=True)
confidence_rows = []
for band in confidence_labels:
    selected = analysis.loc[analysis["confidence_band"] == band]
    n = len(selected)
    confidence_rows.append({
        "confidence_band": band, "rows": n,
        "errors": int((~selected["correct"].astype(bool)).sum()),
        "error_rate": float((~selected["correct"].astype(bool)).mean()) if n else np.nan,
        "FP": int(((selected["y_true"] == 0) & (selected["prediction"] == 1)).sum()),
        "FN": int(((selected["y_true"] == 1) & (selected["prediction"] == 0)).sum()),
    })
confidence_summary = pd.DataFrame(confidence_rows)

member_votes_by_error = pd.crosstab(analysis["error_type"], analysis["member_positive_votes"])
member_votes_by_error = member_votes_by_error.reindex(index=["TN", "FP", "FN", "TP"], columns=[0, 1, 2, 3], fill_value=0)
error_member_summary = (analysis.loc[~analysis["correct"].astype(bool)]
    .groupby("error_type", observed=False)
    .agg(rows=("record_key", "size"), member_disagreement_rate=("member_disagreement", "mean"),
         median_member_probability_range=("member_probability_range", "median"),
         p90_member_probability_range=("member_probability_range", lambda values: values.quantile(0.90)))
    .reset_index())

# 원문/음성은 포함하지 않고, 오류 재검토용 ID·시간·확률만 새 하위 폴더에 저장합니다.
error_ledger_columns = [
    "record_key", "call_id", "y_true", "prediction", "error_type",
    "probability_speaker1", "confidence", "margin_from_0_5", "duration_sec", "overlap",
    "pred_tabm", "pred_resnet", "pred_gandalf", "member_positive_votes", "member_disagreement",
    "p_tabm", "p_resnet", "p_gandalf", "member_probability_min", "member_probability_max",
    "member_probability_range",
]
error_ledger = (analysis.loc[~analysis["correct"].astype(bool), error_ledger_columns]
    .sort_values(["error_type", "confidence"], ascending=[True, False])
    .reset_index(drop=True))
analysis_output_dir = ANALYSIS_RUN_ROOT / "misclassification_analysis"
analysis_output_dir.mkdir(exist_ok=True)
error_summary.to_csv(analysis_output_dir / "e_trg_error_summary.csv", index=False)
error_slices.to_csv(analysis_output_dir / "e_trg_error_slices.csv", index=False)
confidence_summary.to_csv(analysis_output_dir / "e_trg_confidence_bands.csv", index=False)
member_votes_by_error.to_csv(analysis_output_dir / "e_trg_member_votes_by_error.csv")
error_member_summary.to_csv(analysis_output_dir / "e_trg_error_member_summary.csv", index=False)
error_ledger.to_parquet(analysis_output_dir / "e_trg_misclassified_rows.parquet", index=False)
error_ledger.to_csv(analysis_output_dir / "e_trg_misclassified_rows.csv", index=False)
analysis_record = {
    "run_id": ANALYSIS_RUN_ID, "split": "Official Validation", "model": "E_TRG",
    "threshold": 0.5, "rows": len(analysis), "errors": wrong, "fp": fp, "fn": fn,
    "source": "saved parquet predictions + identity-validated text-free Official Validation manifest",
    "audio_listening": False, "transcript_used": False, "retraining_or_refit": False,
    "threshold_or_model_tuning": False,
    "outputs": sorted(path.name for path in analysis_output_dir.iterdir() if path.is_file()),
}
(analysis_output_dir / "analysis_manifest.json").write_text(
    json.dumps(analysis_record, ensure_ascii=False, indent=2), encoding="utf-8")

print(f"Official Validation misclassification analysis: PASS | {ANALYSIS_RUN_ID}")
print(f"Rows={len(analysis):,} | E_TRG errors={wrong:,} | FP={fp:,} | FN={fn:,} | threshold=0.5")
print(f"Saved analysis artifacts: {analysis_output_dir}")
print("Confusion matrix (rows=true speaker 0/1, columns=predicted speaker 0/1):")
display(confusion)
print("Overall E_TRG error summary:")
display(error_summary)
print("FP/FN by the project's existing duration and overlap slices:")
display(error_slices)
print("Error rate by prediction confidence; descriptive only:")
display(confidence_summary)
print("E_TRG error rows by how many of the three members predicted speaker 1:")
display(member_votes_by_error)
print("Within E_TRG errors, member disagreement and probability spread:")
display(error_member_summary)
print("Individual error rows are saved in the run Drive folder; this cell displays aggregate results only.")
print("No audio or transcript was opened; no fitting, extraction, threshold tuning, or model selection was performed.")